# CliffForge — Extended Evaluation on Dablander QSAR-Activity-Cliff MMP Dataset

**What this notebook does:**
1. Installs all dependencies (compatible pinned versions)
2. Clones the CliffForge repository from GitHub
3. Downloads the Dablander `postera_sars_cov_2_mpro` dataset (~390 molecules, ~560 published MMP pairs)
4. Pre-processes it into the `all.csv` format used by CliffForge
5. Trains the Siamese relative-affinity model on the new split
6. Evaluates: within-series Spearman rho, activity-cliff AP/F1/ROC-AUC, cluster-bootstrap CIs
7. Produces side-by-side comparison tables vs the original 1,102-molecule results

**Runtime:** GPU optional (T4 recommended). ~20-30 min on T4, ~60-90 min CPU-only.

In [ ]:
# Cell 0 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU available:')
    print(result.stdout[:400])
else:
    print('No GPU — will run on CPU')

In [ ]:
%%capture
# Cell 1a — Install RDKit
!pip install rdkit-pypi==2022.9.5 -q

In [ ]:
%%capture
# Cell 1b — Install torch-geometric and all other deps
import torch
tv = torch.__version__.split('+')[0]
cu = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'torch={tv}  cuda_suffix={cu}')

!pip install torch-geometric==2.3.1 -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{tv}+{cu}.html -q
!pip install numpy==1.23.5 scipy==1.9.3 pandas==1.5.3 scikit-learn==1.1.3 -q
!pip install chemprop==1.6.1 -q
!pip install matplotlib==3.7.5 tqdm -q
print('Done.')

In [ ]:
# Cell 1c — Verify key imports
import numpy, scipy, pandas, sklearn, rdkit, torch
import torch_geometric
print(f'numpy={numpy.__version__} pandas={pandas.__version__} torch={torch.__version__}')
print(f'torch_geometric={torch_geometric.__version__}')
from rdkit import Chem
print('rdkit OK')

In [ ]:
# Cell 2 — Clone CliffForge repository
import os, sys

REPO_DIR = '/content/TELEMEDICON26'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/chidwipak/TELEMEDICON26.git {REPO_DIR}
else:
    print('Repo already present, pulling...')
    !cd {REPO_DIR} && git pull

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('src/ on path:', src_path)
!ls {REPO_DIR}/src/cliffforge/

In [ ]:
# Cell 3 — Download Dablander QSAR-MMP dataset
import os, urllib.request

DATA_DIR = '/content/dablander_data'
os.makedirs(DATA_DIR, exist_ok=True)

BASE = ('https://raw.githubusercontent.com/MarkusFerdinandDablander/'
        'QSAR-activity-cliff-experiments/main/data/postera_sars_cov_2_mpro')

for fname in ['molecule_data_clean.csv', 'MMP_data_clean.csv']:
    dest = f'{DATA_DIR}/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(f'{BASE}/{fname}', dest)
    else:
        print(f'{fname} already present')
    !wc -l {dest}

In [ ]:
# Cell 4 — Inspect raw data
import pandas as pd

mol_df = pd.read_csv(f'{DATA_DIR}/molecule_data_clean.csv')
print(f'molecule_data_clean: {mol_df.shape}  columns={list(mol_df.columns)}')
print(mol_df.head(3))

mmp_raw = pd.read_csv(f'{DATA_DIR}/MMP_data_clean.csv')
print(f'\nMMP_data_clean: {mmp_raw.shape}  columns={list(mmp_raw.columns)}')
print(mmp_raw.head(2))

In [ ]:
# Cell 5 — Pre-process: Build all.csv in CliffForge format
import math, os, csv, json
from collections import defaultdict
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

MIN_SERIES = 5   # lowered from 8 (smaller dataset)
TEST_FRAC  = 0.35
SEED       = 17

PROCESSED_DIR = '/content/dablander_processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

def canon2d(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None, None
    flat = Chem.MolToSmiles(m, isomericSmiles=False)
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(flat))
        scaf_smi = Chem.MolToSmiles(scaf)
    except Exception:
        scaf_smi = None
    return flat, (scaf_smi or 'NONE')

raw = pd.read_csv(f'{DATA_DIR}/molecule_data_clean.csv')
raw.columns = ['smiles_raw', 'ic50_um']

seen_flat = {}
for _, row in raw.iterrows():
    ic50 = float(row['ic50_um'])
    if ic50 <= 0 or not np.isfinite(ic50):
        continue
    pic50 = 6.0 - math.log10(ic50)
    flat, scaf = canon2d(row['smiles_raw'])
    if flat is None:
        continue
    if flat in seen_flat:
        seen_flat[flat]['vals'].append(pic50)
    else:
        seen_flat[flat] = {'smiles': flat, 'scaf': scaf, 'vals': [pic50]}

records = []
for flat, d in seen_flat.items():
    vals = np.array(d['vals'])
    records.append({'smiles': flat, 'pIC50': float(vals.mean()),
                    'scaf': d['scaf'], 'n_rep': len(vals),
                    'rep_sd': float(vals.std(ddof=1)) if len(vals) > 1 else 0.0})

print(f'Unique 2D structures: {len(records)}')

# Scaffold-based series assignment + within-series split
scaf_members = defaultdict(list)
for rec in records:
    scaf_members[rec['scaf']].append(rec)

rng = np.random.RandomState(SEED)
n_series_used = 0
for scaf, members in scaf_members.items():
    size = len(members)
    if size >= MIN_SERIES:
        n_series_used += 1
        idx = rng.permutation(size)
        n_test = max(2, min(int(round(size * TEST_FRAC)), size - 2))
        test_ids = set(idx[:n_test].tolist())
        for i, rec in enumerate(members):
            rec['series']      = scaf
            rec['series_size'] = size
            rec['split']       = 'test' if i in test_ids else 'train'
    else:
        for rec in members:
            rec['series']      = scaf if size > 1 else 'singleton'
            rec['series_size'] = size
            rec['split']       = 'train'

print(f'Series with >= {MIN_SERIES} members: {n_series_used}')

out_all   = f'{PROCESSED_DIR}/all.csv'
out_train = f'{PROCESSED_DIR}/train.csv'
out_test  = f'{PROCESSED_DIR}/test.csv'

with open(out_all, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['smiles', 'pIC50', 'series', 'series_size', 'split', 'n_rep', 'rep_sd'])
    for rec in records:
        w.writerow([rec['smiles'], f"{rec['pIC50']:.4f}", rec['series'],
                    rec['series_size'], rec['split'], rec['n_rep'], f"{rec['rep_sd']:.4f}"])

for split_name, fpath in [('train', out_train), ('test', out_test)]:
    with open(fpath, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['smiles', 'pIC50'])
        for rec in records:
            if rec['split'] == split_name:
                w.writerow([rec['smiles'], f"{rec['pIC50']:.4f}"])

n_train = sum(1 for r in records if r['split'] == 'train')
n_test  = sum(1 for r in records if r['split'] == 'test')
print(f'all.csv: {len(records)} rows  |  train: {n_train}  |  test: {n_test}')

In [ ]:
# Cell 6 — Pre-process MMP pairs (leakage guard: train-train only)
import pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

all_df = pd.read_csv(out_all)
smiles_to_pic50 = dict(zip(all_df.smiles, all_df.pIC50))
smiles_to_split = dict(zip(all_df.smiles, all_df.split))

def to_flat(smi):
    m = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(m, isomericSmiles=False) if m else None

mmp_raw = pd.read_csv(f'{DATA_DIR}/MMP_data_clean.csv')
mmp_rows, skipped_nf, skipped_leak = [], 0, 0

for _, row in mmp_raw.iterrows():
    s1, s2 = to_flat(row['smiles_1']), to_flat(row['smiles_2'])
    if s1 is None or s2 is None or s1 not in smiles_to_pic50 or s2 not in smiles_to_pic50:
        skipped_nf += 1; continue
    if smiles_to_split[s1] != 'train' or smiles_to_split[s2] != 'train':
        skipped_leak += 1; continue
    mmp_rows.append({'smiles_1': s1, 'smiles_2': s2,
                     'pIC50_1': smiles_to_pic50[s1], 'pIC50_2': smiles_to_pic50[s2],
                     'delta_pIC50': smiles_to_pic50[s1] - smiles_to_pic50[s2]})

mmp_train = pd.DataFrame(mmp_rows)
out_mmp = f'{PROCESSED_DIR}/mmp_train.csv'
mmp_train.to_csv(out_mmp, index=False)
print(f'Raw MMP pairs: {len(mmp_raw)}')
print(f'Skipped (not in processed table): {skipped_nf}')
print(f'Skipped (leakage guard - test molecule): {skipped_leak}')
print(f'Train MMP pairs retained: {len(mmp_train)}')
print(f'delta_pIC50 range: {mmp_train.delta_pIC50.min():.2f} to {mmp_train.delta_pIC50.max():.2f}')

In [ ]:
# Cell 7 — Compute similarity matrix and verify cliff counts
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

all_df  = pd.read_csv(out_all)
n       = len(all_df)
y       = all_df.pIC50.values.astype(float)
is_test = (all_df.split == 'test').values
is_train = ~is_test

print(f'n={n}  train={is_train.sum()}  test={is_test.sum()}')
print('Computing fingerprints...')
fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, 2048)
       for s in all_df.smiles]
S   = np.array([DataStructs.BulkTanimotoSimilarity(f, fps) for f in fps], dtype=np.float32)
print('Done.')

iu, ju = np.triu_indices(n, 1)
sim    = S[iu, ju]
dy     = np.abs(y[iu] - y[ju])
ptype  = np.where(is_test[iu] & is_test[ju], 'test_test',
         np.where(is_train[iu] & is_train[ju], 'train_train', 'test_train'))

print('\nCliff counts:')
for sim_t in (0.8, 0.7, 0.6):
    for d_t in (1.0, 0.5):
        mask = (sim >= sim_t) & (dy >= d_t)
        label = f'Tan>={sim_t} |d|>={d_t}'
        print(f'  {label}: all={mask.sum()}  '
              f'TT={(mask&(ptype=="test_test")).sum()}  '
              f'TR={(mask&(ptype=="test_train")).sum()}  '
              f'RR={(mask&(ptype=="train_train")).sum()}')

primary = (sim >= 0.8) & (dy >= 1.0)

In [ ]:
# Cell 8 — Configure CliffForge paths
import os, sys

src_path = '/content/TELEMEDICON26/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Reimport after path adjustment (in case of stale cache)
import importlib
if 'cliffforge' in sys.modules:
    for mod in list(sys.modules.keys()):
        if mod.startswith('cliffforge'):
            del sys.modules[mod]

MODEL_DIR  = '/content/dablander_models'
OUTPUT_DIR = '/content/dablander_outputs'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ['CLIFFFORGE_DATA']   = PROCESSED_DIR
os.environ['CLIFFFORGE_MODELS'] = MODEL_DIR
os.environ['CLIFFFORGE_OUTPUT'] = OUTPUT_DIR

from cliffforge import paths as P
from pathlib import Path
P._CFG['data']   = Path(PROCESSED_DIR)
P._CFG['models'] = Path(MODEL_DIR)
P._CFG['output'] = Path(OUTPUT_DIR)

print('CliffForge paths configured:')
print(f'  processed_csv: {P.processed_csv()}')
print(f'  model_dir:     {P.model_dir()}')
print(f'  output_dir:    {P.output_dir()}')
print(f'  all.csv exists: {P.processed_csv().exists()}')

In [ ]:
# Cell 9 — Training configuration
# TRAIN_MODE = 'series'  -> train on within-series pairs (CliffForge default)
# TRAIN_MODE = 'mmp'     -> train on Dablander published MMP pairs
TRAIN_MODE = 'mmp'   # <-- change to 'series' to use scaffold-based pairs
EPOCHS = 70
SEEDS  = [0, 1, 2]
print(f'Training mode: {TRAIN_MODE} | epochs: {EPOCHS} | seeds: {SEEDS}')

In [ ]:
# Cell 10 — Train Siamese relative-affinity model
import torch, time, os
import numpy as np
from torch.utils.data import DataLoader
import torch.nn as nn

from cliffforge.relmodel import load_items, NODE_DIM, EDGE_DIM
from cliffforge.networks import SiameseAVES
from cliffforge.pairs import PairSet, collate, make_pairs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')


def make_mmp_pairs(items, mmp_csv):
    """Build training pairs from published MMP table (train partition only)."""
    import pandas as pd
    smi_to_idx = {it['smiles']: k for k, it in enumerate(items) if it['split'] == 'train'}
    df = pd.read_csv(mmp_csv)
    pairs = []
    for _, row in df.iterrows():
        i = smi_to_idx.get(row['smiles_1'])
        j = smi_to_idx.get(row['smiles_2'])
        if i is not None and j is not None and i != j:
            pairs.append((i, j))
            pairs.append((j, i))  # both directions
    return pairs


items = load_items()
print(f'Molecules loaded: {len(items)} '
      f'(train={sum(it["split"]=="train" for it in items)}, '
      f'test={sum(it["split"]=="test" for it in items)})')

for seed in SEEDS:
    print(f'\n========== Training seed {seed} ==========')
    torch.manual_seed(seed)
    np.random.seed(seed)

    if TRAIN_MODE == 'mmp':
        pairs = make_mmp_pairs(items, out_mmp)
        print(f'  MMP pairs: {len(pairs)} (both directions included)')
    else:
        pairs = make_pairs(items, seed)
        print(f'  Series pairs: {len(pairs)}')

    # Leakage check
    assert all(
        items[i]['split'] == 'train' and items[j]['split'] == 'train'
        for i, j in pairs
    ), 'LEAKAGE DETECTED: test molecule in training pairs!'

    loader = DataLoader(PairSet(items, pairs), batch_size=32,
                        shuffle=True, collate_fn=collate)
    model  = SiameseAVES(NODE_DIM, EDGE_DIM, 128, 4, 0.1).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    lossf  = nn.SmoothL1Loss()

    t0 = time.time()
    for ep in range(EPOCHS):
        model.train()
        tot, cnt = 0.0, 0
        for c, r, yb in loader:
            c, r, yb = c.to(DEVICE), r.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = lossf(model(c, r), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot += float(loss) * len(yb); cnt += len(yb)
        if ep % 10 == 0 or ep == EPOCHS - 1:
            print(f'  ep{ep:03d} loss={tot/cnt:.4f}  t={time.time()-t0:.0f}s')

    save_path = f'{MODEL_DIR}/rel_siamese_s{seed}.pt'
    torch.save({'state_dict': model.state_dict(), 'seed': seed,
                'epochs': EPOCHS, 'train_mode': TRAIN_MODE,
                'dataset': 'dablander_postera_sars_cov_2_mpro'}, save_path)
    print(f'  Saved: {save_path}')

print('\nAll seeds trained successfully.')

In [ ]:
# Cell 11 — KNN baseline and Reference-Anchored Relative Model predictions
import numpy as np, pandas as pd, torch
from collections import defaultdict
from cliffforge.relmodel import RelScorer

all_df  = pd.read_csv(out_all)
y       = all_df.pIC50.values.astype(float)
is_train = (all_df.split == 'train').values
is_test  = ~is_train

def knn_predict(S, y_known, is_train_mask, k=5):
    n = len(y_known)
    pred = np.zeros(n)
    tr_idx = np.where(is_train_mask)[0]
    for i in range(n):
        sims = S[i, tr_idx].copy()
        if is_train_mask[i]:
            sims[tr_idx == i] = -1
        top = np.argsort(-sims)[:k]
        w   = np.clip(sims[top], 1e-6, None)
        pred[i] = float(np.sum(w * y_known[tr_idx[top]]) / np.sum(w))
    return pred

ymask = y.copy(); ymask[is_test] = np.nan
knn_pred_all = knn_predict(S, ymask, is_train)

scorer   = RelScorer(model_dir=MODEL_DIR, device=torch.device('cpu'))
n_models = len(scorer.models)
print(f'Loaded {n_models} model checkpoints')

# Reference-anchored predictions for test molecules
by_ser = defaultdict(list)
for i in np.where(is_train)[0]:
    by_ser[all_df.series[i]].append(i)

anch = np.full((n_models, n), np.nan)
for s, g in all_df[is_test].groupby('series'):
    refs = by_ser.get(s, [])
    if not refs:
        continue
    q = list(g.index)
    anch[:, q] = scorer.anchored(list(all_df.smiles[q]),
                                  list(all_df.smiles[refs]), y[refs])
anch_mean = anch.mean(0)

all_df['knn_pred'] = knn_pred_all
all_df['rel_anch'] = anch_mean
print('Predictions computed.')

In [ ]:
# Cell 12 — Within-Series Spearman rho evaluation
import numpy as np, pandas as pd
from scipy.stats import spearmanr, wilcoxon

MIN_SERIES_TEST = 4  # min test molecules per series for Spearman

def within_series_spearman(df, pred_col, min_test=MIN_SERIES_TEST):
    rows = []
    for s, g in df[df.split == 'test'].groupby('series'):
        if len(g) >= min_test and np.isfinite(g[pred_col]).all():
            rho = spearmanr(g['pIC50'], g[pred_col])[0]
            rows.append({'series': s[:50], 'n_test': len(g), 'spearman_rho': rho})
    return pd.DataFrame(rows)

ws_knn = within_series_spearman(all_df, 'knn_pred')
ws_rel = within_series_spearman(all_df, 'rel_anch')

print('=== Within-Series Spearman rho ===')
print(f'5-NN baseline:   n_series={len(ws_knn)}  mean rho = {ws_knn.spearman_rho.mean():.3f}')
print(f'Relative model:  n_series={len(ws_rel)}  mean rho = {ws_rel.spearman_rho.mean():.3f}')

if len(ws_knn) >= 6 and len(ws_rel) >= 6:
    merged = ws_knn.merge(ws_rel, on='series', suffixes=('_knn','_rel'))
    if len(merged) >= 6:
        stat = wilcoxon(merged.spearman_rho_rel.values, merged.spearman_rho_knn.values)
        print(f'Wilcoxon rel vs knn (n={len(merged)} series): p = {stat.pvalue:.4f}')
    print('\nPer-series (relative model):')
    print(ws_rel.sort_values('spearman_rho', ascending=False).to_string(index=False))
else:
    print(f'NOTE: Only {len(ws_rel)} series with >={MIN_SERIES_TEST} test molecules.')
    print('Per-series detail:')
    print(ws_rel.to_string(index=False))

In [ ]:
# Cell 13 — Activity Cliff Detection (AP, F1, ROC-AUC) — both settings
import numpy as np, pandas as pd, json
from sklearn.metrics import average_precision_score, roc_auc_score
import torch

SIM_PRIMARY = 0.80
DELTA       = 1.0
TAU_FIXED   = 1.0
NBOOT       = 500   # increase to 2000 for final publication results

# Collect all pairs (sim >= 0.6) involving at least one test member
m6       = (sim >= 0.6) & (ptype != 'train_train')
pairs_h  = iu[m6]
pairs_r  = ju[m6]
pt6      = ptype[m6]

# For Setting B, query=test, ref=train
pairs_hh, pairs_rr, kind_arr, pt_arr = [], [], [], []
for i, j, pt in zip(pairs_h, pairs_r, pt6):
    if pt == 'test_test':
        pairs_hh.append(i); pairs_rr.append(j); kind_arr.append('AA')
    else:  # test_train -> query=test member
        h, r = (i, j) if is_test[i] else (j, i)
        pairs_hh.append(h); pairs_rr.append(r); kind_arr.append('BQ')
    pt_arr.append(pt)

H   = np.array(pairs_hh)
Rr  = np.array(pairs_rr)
pt_arr = np.array(pt_arr)
kind_arr = np.array(kind_arr)

# Relative model predictions (structure only, no measured activity needed)
print(f'Scoring {len(H)} pairs with relative model...')
rel_all  = scorer.pair_delta_all(list(all_df.smiles[H]), list(all_df.smiles[Rr]))
rel_mean = rel_all.mean(0)

# KNN: for B pairs use measured training reference
isB      = kind_arr == 'BQ'
knn_main = knn_pred_all[H] - knn_pred_all[Rr]
knn_main[isB] = knn_pred_all[H[isB]] - y[Rr[isB]]  # use actual measured ref

tsim   = S[H, Rr]
true_d = y[H] - y[Rr]

all_results = {}

print('\n' + '='*70)
print('ACTIVITY CLIFF DETECTION — Dablander MMP dataset')
print('='*70)

for setting, setting_label in [('test_test', 'A (strict test-test)'),
                                 ('test_train', 'B (lead-opt test-train)')]:
    for sim_t in (0.8, 0.7, 0.6):
        mask   = (pt_arr == setting) & (tsim >= sim_t)
        idx    = np.where(mask)[0]
        if len(idx) == 0:
            continue
        ytrue  = (np.abs(true_d[idx]) >= DELTA).astype(int)
        n_pos  = ytrue.sum()
        key    = f'{setting}|sim>={sim_t}'

        res = {'n_pairs': len(idx), 'n_cliffs': int(n_pos),
               'prevalence': float(n_pos/len(idx)) if len(idx) else None}

        print(f'\n[{setting_label} | sim>={sim_t}] '
              f'n_pairs={len(idx)}  n_cliffs={n_pos}  '
              f'prevalence={100*n_pos/len(idx):.1f}%')

        for mname, sc in [('rel_siamese', np.abs(rel_mean[idx])),
                           ('knn_sim',     np.abs(knn_main[idx]))]:
            r = {}
            if n_pos > 0 and n_pos < len(ytrue):
                r['AP']      = float(average_precision_score(ytrue, sc))
                r['ROC_AUC'] = float(roc_auc_score(ytrue, sc))
            else:
                r['AP'] = r['ROC_AUC'] = None
            yh  = (sc >= TAU_FIXED).astype(int)
            tp  = int(((ytrue==1)&(yh==1)).sum())
            fp  = int(((ytrue==0)&(yh==1)).sum())
            fn  = int(((ytrue==1)&(yh==0)).sum())
            r['F1_tau1.0'] = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
            r['tp'] = tp; r['fp'] = fp; r['fn'] = fn
            res[mname] = r
            ap_str = f"{r['AP']:.3f}" if r['AP'] is not None else 'N/A (no variation)'
            print(f'  {mname:<14} AP={ap_str}  '
                  f'F1@tau=1={r["F1_tau1.0"]:.3f}  '
                  f'tp={tp} fp={fp} fn={fn}')

        all_results[key] = res

# Save
def _default(o):
    if isinstance(o, float) and not (o == o): return None
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    return str(o)
json.dump(all_results, open(f'{OUTPUT_DIR}/cliff_detection_metrics.json', 'w'), indent=2, default=_default)
print(f'\nSaved to {OUTPUT_DIR}/cliff_detection_metrics.json')

In [ ]:
# Cell 14 — Cluster Bootstrap CIs for AP difference
import numpy as np
from collections import defaultdict
from sklearn.metrics import average_precision_score

def components(pairs_ab):
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a, b in pairs_ab:
        parent[find(a)] = find(b)
    return [find(a) for a, _ in pairs_ab]

def cluster_boot(cluster_ids, evalfn, nboot=NBOOT, seed=0):
    rng = np.random.RandomState(seed)
    cl  = defaultdict(list)
    for i, c in enumerate(cluster_ids):
        cl[c].append(i)
    keys = list(cl)
    return [evalfn(np.array([j for p in rng.randint(0, len(keys), len(keys)) for j in cl[keys[p]]], dtype=int))
            for _ in range(nboot)]

def ci95(vals):
    v = np.array([x for x in vals if x is not None and np.isfinite(x)])
    if len(v) < 20:
        return None, None, len(v)
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)), len(v)

print(f'Cluster bootstrap (NBOOT={NBOOT}) for primary setting (sim>=0.8)...')
print()

for setting, cunit_label in [('test_test', 'connected_components'),
                               ('test_train', 'query_molecule')]:
    mask  = (pt_arr == setting) & (tsim >= SIM_PRIMARY)
    idx   = np.where(mask)[0]
    if len(idx) == 0:
        print(f'{setting}: no pairs at sim>={SIM_PRIMARY}'); continue
    ytrue = (np.abs(true_d[idx]) >= DELTA).astype(int)
    if ytrue.sum() == 0 or ytrue.sum() == len(ytrue):
        print(f'{setting}: cannot compute AP (all pos or all neg — only {ytrue.sum()} cliffs of {len(idx)} pairs)'); continue

    sr = np.abs(rel_mean[idx])
    sk = np.abs(knn_main[idx])

    if setting == 'test_test':
        cluster_ids = components([(int(H[i]), int(Rr[i])) for i in idx])
    else:
        cluster_ids = [int(H[i]) for i in idx]

    def ev(ix):
        yt = ytrue[ix]
        if 0 < yt.sum() < len(yt):
            return float(average_precision_score(yt, sr[ix])), float(average_precision_score(yt, sk[ix]))
        return np.nan, np.nan

    boots = cluster_boot(cluster_ids, ev, NBOOT)
    d_vals = [(a-b) if np.isfinite(a) and np.isfinite(b) else None for a, b in boots]
    lo_d, hi_d, nv = ci95(d_vals)

    pa_rel = float(average_precision_score(ytrue, sr))
    pa_knn = float(average_precision_score(ytrue, sk))

    print(f'--- {setting} | n_pairs={len(idx)} n_cliffs={ytrue.sum()} cluster_unit={cunit_label} ---')
    print(f'  rel_siamese AP = {pa_rel:.3f}')
    print(f'  knn_sim     AP = {pa_knn:.3f}')
    if lo_d is not None:
        print(f'  Delta AP (rel-knn) = {pa_rel-pa_knn:+.3f}  95% CI [{lo_d:.3f}, {hi_d:.3f}]  (n_valid_boots={nv})')
        if lo_d > 0:
            print(f'  >>> CI excludes 0: STATISTICALLY SIGNIFICANT improvement at 95%')
        else:
            print(f'  >>> CI includes 0: not significant at 95%')
    else:
        print(f'  Not enough valid bootstrap resamples for CI (n_valid={nv})')
    print()

In [ ]:
# Cell 15 — Summary comparison table
import pandas as pd

n_cliffs_tt = int((primary & (ptype == 'test_test')).sum())
n_cliffs_all = int(primary.sum())

rows = [
    ['Dataset source', 'COVID Moonshot CDD raw table', 'Dablander QSAR-MMP repo'],
    ['N molecules', '1,102', str(n)],
    ['N train / N test', '961 / 141', f'{int(is_train.sum())} / {int(is_test.sum())}'],
    ['Published MMP pairs (train)', '—', str(len(mmp_train))],
    ['Primary cliffs (all)', '46', str(n_cliffs_all)],
    ['Primary cliffs (test-test)', '5', str(n_cliffs_tt)],
    ['N series for Spearman', '12', str(len(ws_rel))],
    ['Within-series Spearman (KNN)', '~0.19', f'{ws_knn.spearman_rho.mean():.3f}' if len(ws_knn) else 'N/A'],
    ['Within-series Spearman (rel)', '0.581 (p=0.005)', f'{ws_rel.spearman_rho.mean():.3f}' if len(ws_rel) else 'N/A'],
    ['Setting A AP (rel / knn)', '0.731 / 0.331', 'see Cell 14 above'],
    ['Setting A CI excludes 0?', 'No (CI -0.03 to +0.70)', 'see Cell 14 above'],
]

cmp_df = pd.DataFrame(rows, columns=['Metric', 'Original (1102 mol)', 'Dablander (this run)'])
print(cmp_df.to_string(index=False))

In [ ]:
# Cell 16 — Save all results and download
import json, os, shutil
import numpy as np

def _default(o):
    if isinstance(o, float) and not np.isfinite(o): return None
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

summary = {
    'dataset': 'Dablander postera_sars_cov_2_mpro',
    'train_mode': TRAIN_MODE,
    'n_molecules': n, 'n_train': int(is_train.sum()), 'n_test': int(is_test.sum()),
    'mmp_train_pairs': len(mmp_train),
    'cliff_definition': {'tanimoto': SIM_PRIMARY, 'abs_delta_pIC50': DELTA},
    'cliffs': {'all': n_cliffs_all, 'test_test': n_cliffs_tt},
    'within_series_spearman': {
        'n_series': len(ws_rel),
        'knn_mean': float(ws_knn.spearman_rho.mean()) if len(ws_knn) else None,
        'rel_mean': float(ws_rel.spearman_rho.mean()) if len(ws_rel) else None,
    },
    'cliff_detection': all_results,
    'original_reference': {
        'n_molecules': 1102, 'cliffs_test_test': 5,
        'within_series_spearman_rel': 0.581,
        'setting_A_AP_rel': 0.731,
    },
}

json.dump(summary, open(f'{OUTPUT_DIR}/dablander_full_summary.json', 'w'), indent=2, default=_default)
print(f'Results saved in: {OUTPUT_DIR}')

try:
    from google.colab import files
    zip_path = '/content/cliffforge_dablander_results'
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
    files.download(zip_path + '.zip')
    print('Download triggered.')
except ImportError:
    print('Not running in Colab — results in:', OUTPUT_DIR)